## **Malay-English Hallucination Detection**

This notebook implements a supervised NLP classification model to detect truthful and hallucinated answers using a Malay-English HaluEval dataset.

###### 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re

from google.colab import files

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix

from scipy.sparse import hstack, csr_matrix

###### 2. Upload and Load Dataset

In [2]:
uploaded = files.upload()

df = pd.read_csv("englishmalay_halueval.csv")

df.head()

Saving englishmalay_halueval.csv to englishmalay_halueval.csv


,id,language,question,answer,label,label_text,source_dataset,creation_method,context
0,1,English,Which magazine was started first Arthur's Maga...,Arthur's Magazine,0,truthful,HaluEval,original,Arthur's Magazine (1844–1846) was an American ...
1,2,English,Which magazine was started first Arthur's Maga...,First for Women was started first.,1,hallucinated,HaluEval,original,Arthur's Magazine (1844–1846) was an American ...
2,3,English,The Oberoi family is part of a hotel company t...,Delhi,0,truthful,HaluEval,original,The Oberoi family is an Indian family that is ...
3,4,English,The Oberoi family is part of a hotel company t...,The Oberoi family's hotel company is based in ...,1,hallucinated,HaluEval,original,The Oberoi family is an Indian family that is ...
4,5,English,Musician and satirist Allie Goertz wrote a son...,President Richard Nixon,0,truthful,HaluEval,original,"Allison Beth ""Allie"" Goertz (born March 2, 199..."


###### 3. Dataset Checking

In [3]:
print("Dataset shape:", df.shape)

print("\nLanguage distribution:")
print(df["language"].value_counts())

print("\nLabel distribution:")
print(df["label"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

Dataset shape: (1000, 9)

Language distribution:
language
English    500
Malay      500
Name: count, dtype: int64

Label distribution:
label
0    500
1    500
Name: count, dtype: int64

Missing values:
id                 0
language           0
question           0
answer             0
label              0
label_text         0
source_dataset     0
creation_method    0
context            0
dtype: int64


###### 4. Data Cleaning

In [4]:
df = df.dropna(subset=["question", "answer", "label", "context"])
df = df.drop_duplicates(subset=["question", "answer", "label", "language"])

print("Dataset shape after cleaning:", df.shape)

Dataset shape after cleaning: (1000, 9)


###### 5. Combine Context, Question and Answer

In [5]:
df["text"] = (
    "Context: " + df["context"].astype(str) +
    " Question: " + df["question"].astype(str) +
    " Answer: " + df["answer"].astype(str)
)

df[["text", "label"]].head()

,text,label
0,Context: Arthur's Magazine (1844–1846) was an ...,0
1,Context: Arthur's Magazine (1844–1846) was an ...,1
2,Context: The Oberoi family is an Indian family...,0
3,Context: The Oberoi family is an Indian family...,1
4,"Context: Allison Beth ""Allie"" Goertz (born Mar...",0


###### 6. Create Context-Based Features

In [6]:
def get_words(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\u00C0-\u024F\u1E00-\u1EFF0-9\s]", " ", text)
    words = text.split()
    return set(words)

df["answer_length"] = df["answer"].astype(str).apply(lambda x: len(x.split()))
df["question_length"] = df["question"].astype(str).apply(lambda x: len(x.split()))
df["context_length"] = df["context"].astype(str).apply(lambda x: len(x.split()))

def overlap_ratio(answer, context):
    answer_words = get_words(answer)
    context_words = get_words(context)

    if len(answer_words) == 0:
        return 0

    overlap = answer_words.intersection(context_words)
    return len(overlap) / len(answer_words)

df["answer_context_overlap"] = df.apply(
    lambda row: overlap_ratio(row["answer"], row["context"]),
    axis=1
)

df["answer_in_context"] = df.apply(
    lambda row: 1 if str(row["answer"]).lower() in str(row["context"]).lower() else 0,
    axis=1
)

df[[
    "answer_length",
    "question_length",
    "context_length",
    "answer_context_overlap",
    "answer_in_context"
]].head()

,answer_length,question_length,context_length,answer_context_overlap,answer_in_context
0,2,11,29,1.0,1
1,6,11,29,0.8,0
2,1,17,32,1.0,1
3,9,17,32,0.7,0
4,3,20,70,1.0,1


###### 7. Define Input Features and Target Label

In [7]:
X_text = df["text"]

X_numeric = df[[
    "answer_length",
    "question_length",
    "context_length",
    "answer_context_overlap",
    "answer_in_context"
]]

y = df["label"]

###### 8 . Train/test split

In [8]:
X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text,
    X_numeric,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_text_train))
print("Testing samples:", len(X_text_test))

print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())

Training samples: 800
Testing samples: 200

Training label distribution:
label
1    400
0    400
Name: count, dtype: int64

Testing label distribution:
label
1    100
0    100
Name: count, dtype: int64


###### 9. TF-IDF Feature Representation

In [9]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    lowercase=True,
    ngram_range=(1, 2)
)

X_train_tfidf = vectorizer.fit_transform(X_text_train)
X_test_tfidf = vectorizer.transform(X_text_test)

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF test shape:", X_test_tfidf.shape)

TF-IDF train shape: (800, 10000)
TF-IDF test shape: (200, 10000)


###### 10. Combine TF-IDF and Numeric Features

In [10]:
X_train_num = csr_matrix(X_num_train.values)
X_test_num = csr_matrix(X_num_test.values)

X_train_final = hstack([X_train_tfidf, X_train_num])
X_test_final = hstack([X_test_tfidf, X_test_num])

print("Final training feature shape:", X_train_final.shape)
print("Final testing feature shape:", X_test_final.shape)

Final training feature shape: (800, 10005)
Final testing feature shape: (200, 10005)


###### 11. Train Logistic Regression Model

In [11]:
model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_final, y_train)

print("Model training completed.")

Model training completed.


###### 12. Make Predictions

In [12]:
y_pred = model.predict(X_test_final)

print("Prediction completed.")

Prediction completed.


###### 13. Evaluate Model Performance

In [13]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Model Evaluation Results")
print("------------------------")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1-score :", f1)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Truthful", "Hallucinated"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Model Evaluation Results
------------------------
Accuracy : 0.985
Precision: 0.98989898989899
Recall   : 0.98
F1-score : 0.9849246231155779

Classification Report:
              precision    recall  f1-score   support

    Truthful       0.98      0.99      0.99       100
Hallucinated       0.99      0.98      0.98       100

    accuracy                           0.98       200
   macro avg       0.99      0.98      0.98       200
weighted avg       0.99      0.98      0.98       200


Confusion Matrix:
[[99  1]
 [ 2 98]]
